# Pipeline EHBG-FACS · 04 · NCO por RL (POMO + Attention Model)

**Paradigma 4 — Attention Model entrenado por REINFORCE estilo POMO (GPU).**

Política neuronal entrenada **sin etiquetas**, con el costo de la ruta como recompensa y la estrategia **POMO**: N trayectorias desde nodos de inicio distintos + media como **línea base compartida**. Recompensa = costo **determinista** (tiempo nominal τ) → por eso es 'NCO determinista' y, evaluada bajo ξ, exhibe fragilidad ante la estocasticidad.

In [ ]:
# === Configuración del entorno (ejecuta esta celda primero) =================
# Requiere: (a) el paquete `svrplab` (carpeta experiments/colab del repo de tesis)
#           (b) el repo oficial de SVRPBench (se clona solo en bootstrap.init()).
REPO_URL  = "https://github.com/AbrahaHub/TESIS-ANT"   # <-- EDITA si tu repo difiere
USE_DRIVE = True   # persistir banco/resultados/modelos en Google Drive (recomendado)

import os, sys, subprocess

if USE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except Exception as e:
        print("Drive no disponible (¿ejecutas local?):", e)

def _find_svrplab():
    cands = ["/content/drive/MyDrive/TESIS-ANT/experiments/colab",
             "/content/TESIS-ANT/experiments/colab",
             os.path.join(os.getcwd(), "experiments", "colab"),
             os.getcwd()]
    for c in cands:
        if os.path.isdir(os.path.join(c, "svrplab")):
            return c
    return None

_path = _find_svrplab()
if _path is None:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, "/content/TESIS-ANT"], check=False)
    _path = "/content/TESIS-ANT/experiments/colab"
sys.path.insert(0, _path)
print("svrplab en:", _path)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "numpy", "scipy", "pandas",
                "matplotlib", "scikit-learn", "pillow", "tqdm"], check=False)
# torch ya viene en Colab. gurobipy solo se instala en el notebook del paradigma 1.

from svrplab import bootstrap, protocol, data, runner, metrics, viz
env   = bootstrap.init()        # GPU + repo oficial SVRPBench + rutas (Drive si está montado)
proto = protocol.DEFAULT
print("device:", env.device, "| raíz de artefactos:", env.paths.root)

In [ ]:
# === Configuración del experimento (IDÉNTICA en los 5 notebooks) ============
# Para garantizar el "piso parejo", TODOS los notebooks deben usar los MISMOS
# SIZES y N_INSTANCES: así resuelven exactamente el mismo banco de instancias.
SIZES       = [10, 20, 50]           # clientes. Extiende a [50,100,200,300] (ver notas).
N_INSTANCES = proto.instances_per_size   # 30 (rigor estadístico). Corrida rápida: pon 5.

bank = data.load_bank(env.paths.instances, SIZES, N_INSTANCES,
                      base_seed=proto.base_seed, capacity_mode=proto.capacity_mode, verbose=True)
print({s: len(v) for s, v in bank.items()}, "instancias por tamaño")

## Entrenar (POMO, GPU) e inferir
Entrenamiento autoregresivo con bonus de entropía y AMP en GPU. Aumenta `steps_per_size` para mayor calidad (más exigente en cómputo).

In [ ]:
from svrplab.solvers.nco_rl import NCOReinforce
rl = NCOReinforce(train_sizes=(10,20), steps_per_size=1500, batch=64, embed_dim=128,
                  device=env.device, models_dir=env.paths.models, verbose=True)
df = runner.run_solver(rl, "nco-rl", bank, env, proto, verbose=True)
df

## Curva de entrenamiento y figuras

In [ ]:
import matplotlib.pyplot as plt
if getattr(rl, "history", None):
    viz.plot_training_curve(rl.history, ylabel="costo medio", title="nco-rl (POMO): costo"); plt.show()
display(metrics.aggregate_by_size(df))
inst = bank[SIZES[0]][0]
sol = rl.solve(inst, num_realizations=proto.realizations)
viz.plot_routes(inst, sol.routes, title=f"nco-rl (POMO) · n={SIZES[0]}"); plt.show()

**Interpretación.** Bien entrenado, POMO se acerca al óptimo de costo y **supera a la NCO supervisada** (coherente con la literatura: RL > supervisado), pero al entrenarse en el problema determinista colapsa en factibilidad bajo la estocasticidad de las ventanas.